In [1]:
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import logging
from datetime import datetime

# model_name = "microsoft/mdeberta-v3-base"
model_name = "aubmindlab/bert-base-arabertv02"
EVAL_SIZE = 2000
BATCH_SIZE = 16  # can increase with AMP
GRAD_ACCUM_STEPS = 2  # effective batch = 32
EPOCHS = 5
LR = 2e-5
# LR = 2e-3
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
SEED = 0
MAX_LENGTH = 512
NUM_DROPOUTS = 5
DROPOUT_RATE = 0.1
LABEL_SMOOTHING = 0.1
LLRD_DECAY = 0.95  # layer-wise learning rate decay
NUM_WORKERS = 4
USE_AMP=False
random.seed(SEED)

In [2]:
# after doing some research, i saw this paper https://www.mdpi.com/2079-3197/13/1/5
# which basically showing how mdeberta v3 outperformed every other model even the arabic sepcialized ones like arabert and camelbert
# and clearly arabert outperforms camelbert
# and also here is my theory for why i'm favoring a multilingual model over an arabic one
# labels are in english, and i want them to stay in english, for a model trained on both arabic and english
# it might be easier to grasp the textual understanding of both the input text and the label.

# quoted from the paper
# Although BERT and XLM-RoBERTa employ self-attention mechanisms, 
# mDeBERTa-v3 may incorporate more advanced attention mechanisms, such as disentangled attention, 
# which is specifically designed to capture precise linguistic dependencies. Consequently, 
# this model can surpass the others in tasks requiring extensive linguistic analysis, such as TCU.

In [3]:
train = pd.read_csv("/kaggle/input/eacl-2026-abjad-nlp-shared-task-medical-text-classification-in-arabic/shared_task_train.csv")
test = pd.read_csv("/kaggle/input/eacl-2026-abjad-nlp-shared-task-medical-text-classification-in-arabic/shared_task_devtest_no_label.csv")
train

,text,category,label
0,السؤال\n-------\nالسلام عليكم انا مصاب بفقر ال...,Hematological diseases,33
1,السؤال\n-------\nانا شاب عندى 25 سنه وعندى تبو...,Urogenital diseases,76
2,السؤال\n-------\nصباح الخير عندي القضيب غير نش...,Medicinal herbs,45
3,السؤال\n-------\nهل يظهر الحشيش في تحليل CBC و...,Addiction,0
4,السؤال\n-------\nوزني 58 كغم واريد ان افقد 5 ك...,Biology,7
...,...,...,...
27946,السؤال\n-------\nهل تحليل صوره الدم كامله يبين...,Hematological diseases,33
27947,"السؤال\n-------\n""""""اعاني من حساسية الجلد فعند...",Allergy,1
27948,السؤال\n-------\nاشعر على الدوام ان قلبي يغلي ...,Psychiatric diseases,65
27949,السؤال\n-------\nهناك بروز في الأسنان الأمامية...,Dental diseases,13


In [4]:
print(train['text'].iloc[0])
print(f"category {train['category'].iloc[0]}")
print('='*50)
print(train['text'].iloc[1])
print(f"category {train['category'].iloc[1]}")
print('='*50)
print(train['text'].iloc[2])
print(f"category {train['category'].iloc[2]}")
print('='*50)
print(train['text'].iloc[3])
print(f"category {train['category'].iloc[3]}")
print('='*50)
print(train['text'].iloc[4])
print(f"category {train['category'].iloc[4]}")
print('='*50)
print(train['text'].iloc[5])
print(f"category {train['category'].iloc[5]}")

السؤال
-------
السلام عليكم انا مصاب بفقر الدم المنجلي (السكلسل) علمآ بأن نسبة السكلسل 72 فعندما تصبح نسبة الدم 7 فأن الالام تأتي بكثره فما الحل لزيادة نسبة الدم وما الحل لعلاج...

الجواب
-------
الحل بالابتعاد عن الرضرض النفسية وتقوية المناعة وتناول حمية غذائية متوازنة غنية بالحديد وعند حدوث نوبات الام سببها ونقص حاد بالخضاب الدموي لايوجد الا تعويض الدم الناقص..بنقل الدم.
category Hematological diseases
السؤال
-------
انا شاب عندى 25 سنه وعندى تبول لا ارادى ونا نايم يعنى بحلم انى فى الحمام وفجاه استيقظ على انى على السرير اخر مره حصلت لى فى شهر 7 الماضى...

الجواب
-------
حاول الاستيقاظ ليلا اي في الفترة التي عادة تبول فيها في الفراش . مثلا اذا كان ذلك بعد ذهابك للنوم بساعتين هذا يعني انه عليك الاستيقاظ بعد اقل من ساعتين من نومك للذهاب الى الحمام . واعد الكرة يوميا لفترة
وحاول ان لا تتناول سوائل ساعة قبل النوم
category Urogenital diseases
السؤال
-------
صباح الخير عندي القضيب غير نشط خمول متزوج منذ 17 سنه حيامن ﻻتوجد ممكن الحل

الجواب
-------
بسم الله الرحمن الرحيم
يمكنك متابعتى على ال

In [5]:
label2desc = (
    train[["label", "category"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["category"]
    .to_dict()
)
desc2label = {v: k for k, v in label2desc.items()}
desc2label

{'Addiction': 0,
 'Allergy': 1,
 'Alternative medicine': 2,
 'Anatomy': 3,
 'Anesthesiology': 4,
 'Benign and malignant tumors': 5,
 'Biochemistry': 6,
 'Biology': 7,
 'Cardiothoracic surgery': 8,
 'Cardiovascular diseases': 9,
 'Chemistry': 10,
 'Child health': 11,
 'Congenital malformations': 12,
 'Dental diseases': 13,
 'Dental health': 14,
 'Dentistry': 15,
 'Dermatological diseases': 16,
 'Diabetes': 17,
 'Diagnosis': 18,
 'Ear, nose, and throat (ENT)': 19,
 'Embryology': 20,
 'Endocrine diseases': 21,
 'Eye diseases': 22,
 'First aid': 23,
 'Gastrointestinal diseases': 24,
 'General medicine': 25,
 'General surgery': 26,
 'Genetic diseases': 27,
 'Genetics': 28,
 'Geriatric health': 29,
 'Gynecologic surgery': 30,
 'Gynecological diseases': 31,
 'Health and sports': 32,
 'Hematological diseases': 33,
 'History of medicine': 34,
 'Hormones': 35,
 'Hypertension': 36,
 'Immunology': 37,
 'In vitro fertilization (IVF)': 38,
 'Infectious diseases': 39,
 'Infertility': 40,
 'Internal m

In [6]:
test.head()

,Id,text,Predicted
0,0,السؤال\n-------\nما هو العلاج الأمثل النمش ؟\n...,NaN
1,1,السؤال\n-------\nهل خلل الكرومسومات للجنين له ...,NaN
2,2,السؤال\n-------\nزوجتي حامل في نهاية شهرها الس...,NaN
3,3,السؤال\n-------\nأشعر بإلتهاب أو حروق مستمر في...,NaN
4,4,السؤال\n-------\nارغب بوصفه علاج للضغط ولديه س...,NaN


In [7]:
print(len(train['category'].unique()))
print(train['category'].unique())
print(train.category.value_counts())

82
['Hematological diseases' 'Urogenital diseases' 'Medicinal herbs'
 'Addiction' 'Biology' "Women's health" 'Eye diseases' 'Anesthesiology'
 'Neurological diseases' 'Orthopedic surgery' 'Cardiovascular diseases'
 'Musculoskeletal and joint diseases' 'Respiratory diseases'
 'Psychiatric diseases' 'General surgery' 'Diabetes' 'Child health'
 'Pharmacology' 'Hypertension' 'Dental diseases' 'Sexual health'
 "Men's health" 'Benign and malignant tumors' 'Plastic surgery'
 'Vitamins and minerals' 'Physiotherapy' 'Anatomy' 'Public health'
 'Neurosurgery' 'Skin and beauty' 'Pregnancy and childbirth' 'Infertility'
 'Endocrine diseases' 'Nutrition' 'Immunology' 'General medicine'
 'Oral diseases' 'Sexually transmitted diseases' 'Gynecologic surgery'
 'Dental health' 'Mental health' 'Ear, nose, and throat (ENT)'
 'Cardiothoracic surgery' 'Rheumatic diseases' 'Alternative medicine'
 'Internal medicine diseases' 'Gastrointestinal diseases' 'Allergy'
 'Infectious diseases' 'Dentistry' 'Dermatologica

In [8]:
class CustomDataset(Dataset):
    def __init__(self, tokenizer_name, texts, labels=None, max_length=256):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.texts = texts
        self.labels = labels
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True
        )
        input_dict = {
            'input_ids': torch.tensor(encoded['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoded['attention_mask'], dtype=torch.long),
        }
        # token_type_ids not used by mdeberta
        if 'token_type_ids' in encoded:
            input_dict['token_type_ids'] = torch.tensor(encoded['token_type_ids'], dtype=torch.long)

        if self.labels is not None:
            input_dict['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_dict

def apply_dynamic_padding(inputs):
    """Truncate the text to the max length in the batch to make training faster"""
    mask_len = int(inputs["attention_mask"].sum(axis=1).max())
    for k, v in inputs.items():
        if k != 'labels' and v.dim() == 2:  # skip 1D tensors like labels
            inputs[k] = inputs[k][:, :mask_len]
    return inputs

def collate_fn(batch):
    """Collate with dynamic padding"""
    input_ids = torch.stack([b['input_ids'] for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])

    inputs = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
    }

    if 'token_type_ids' in batch[0]:
        inputs['token_type_ids'] = torch.stack([b['token_type_ids'] for b in batch])

    if 'labels' in batch[0]:
        inputs['labels'] = torch.stack([b['labels'] for b in batch])

    # apply dynamic padding
    inputs = apply_dynamic_padding(inputs)
    return inputs

In [9]:
class AttentionPooling(nn.Module):
    """Attention-weighted pooling over sequence"""
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1, bias=False)
        )
        self._init_weights()

    def _init_weights(self):
        for module in self.attention:
            if isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, hidden_states, attention_mask):
        # hidden_states: (batch, seq_len, hidden)
        # attention_mask: (batch, seq_len)
        attn_scores = self.attention(hidden_states).squeeze(-1)  # (batch, seq_len)
        # use large negative instead of -inf to avoid NaN in softmax
        attn_scores = attn_scores.masked_fill(attention_mask == 0, -1e4)
        attn_weights = F.softmax(attn_scores, dim=1).unsqueeze(-1)  # (batch, seq_len, 1)
        pooled = (hidden_states * attn_weights).sum(dim=1)  # (batch, hidden)
        return pooled


class MeanPooling(nn.Module):
    """Mean pooling over non-padded tokens"""
    def forward(self, hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        return pooled


class MultiSampleDropout(nn.Module):
    """Apply multiple dropout masks with varying rates and average predictions"""
    def __init__(self, hidden_size, num_classes, num_dropouts=5, dropout_rate=0.1):
        super().__init__()
        # varying dropout rates for more diverse predictions
        rates = [dropout_rate + i * 0.05 for i in range(num_dropouts)]  # e.g., [0.1, 0.15, 0.2, 0.25, 0.3]
        self.dropouts = nn.ModuleList([nn.Dropout(r) for r in rates])
        self.classifier = nn.Linear(hidden_size, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.orthogonal_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, x):
        if self.training:
            # average logits from multiple dropout masks
            logits = torch.stack([self.classifier(drop(x)) for drop in self.dropouts], dim=0)
            return logits.mean(dim=0)
        else:
            return self.classifier(x)

            
class MedicalClassifier(nn.Module):
    def __init__(self, model_name, num_classes, num_dropouts=5, dropout_rate=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        
        hidden_size = self.encoder.config.hidden_size

        # pooling layers
        self.attention_pool = AttentionPooling(hidden_size)
        self.mean_pool = MeanPooling()

        # classification head with layer norm and residual-style connection
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dense = nn.Linear(hidden_size * 2, hidden_size)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout_rate)

        # multi-sample dropout classifier
        self.classifier = MultiSampleDropout(hidden_size, num_classes, num_dropouts, dropout_rate)

        self._init_weights()

    def _init_weights(self):
        nn.init.orthogonal_(self.dense.weight)
        nn.init.zeros_(self.dense.bias)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        hidden_states = outputs.last_hidden_state

        # combine attention pooling and mean pooling
        attn_pooled = self.attention_pool(hidden_states, attention_mask)
        mean_pooled = self.mean_pool(hidden_states, attention_mask)
        pooled = torch.cat([attn_pooled, mean_pooled], dim=-1)

        # classification head
        x = self.layer_norm(pooled)
        x = self.dense(x)
        x = self.activation(x)
        x = self.dropout(x)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTHING)

        return {"loss": loss, "logits": logits}

In [10]:
# train_df, eval_df = train_test_split(
#     train,
#     test_size=EVAL_SIZE,
#     stratify=train["label"],
#     random_state=SEED
# )
# train_df = train_df.reset_index(drop=True)
# eval_df = eval_df.reset_index(drop=True)
# print(f"train: {len(train_df)} | eval: {len(eval_df)}")
# print(f"train labels: {train_df['label'].nunique()} | eval labels: {eval_df['label'].nunique()}")

In [11]:
train_ds = CustomDataset(
    model_name,
    # train_df['text'].tolist(),
    # train_df['label'].tolist(),
    train['text'].tolist(),
    train['label'].tolist(),
    max_length=MAX_LENGTH
)
eval_ds = CustomDataset(
    model_name,
    # eval_df['text'].tolist(),
    # eval_df['label'].tolist(),
    train['text'].tolist(),
    train['label'].tolist(),
    max_length=MAX_LENGTH
)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)
eval_loader = DataLoader(
    eval_ds,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
num_labels = len(label2desc)
model = MedicalClassifier(
    model_name,
    num_labels,
    num_dropouts=NUM_DROPOUTS,
    dropout_rate=DROPOUT_RATE
).cuda()

2025-12-19 23:12:23.580618: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766185943.728185      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766185943.771859      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766185944.130226      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766185944.130269      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766185944.130271      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

In [13]:
# optimizer with layer-wise learning rate decay (LLRD)
def get_optimizer_params(model, lr, weight_decay, llrd_decay):
    no_decay = ['bias', 'LayerNorm.weight', 'layer_norm.weight']
    optimizer_params = []
    
    # encoder layers with LLRD
    num_layers = model.encoder.config.num_hidden_layers
    for layer_idx in range(num_layers):
        layer_lr = lr * (llrd_decay ** (num_layers - layer_idx - 1))
        layer_params = [
            (n, p) for n, p in model.encoder.named_parameters()
            if f'layer.{layer_idx}.' in n
        ]
        optimizer_params.append({
            'params': [p for n, p in layer_params if not any(nd in n for nd in no_decay)],
            'lr': layer_lr,
            'weight_decay': weight_decay
        })
        optimizer_params.append({
            'params': [p for n, p in layer_params if any(nd in n for nd in no_decay)],
            'lr': layer_lr,
            'weight_decay': 0.0
        })
    
    # embeddings (lowest LR)
    embed_lr = lr * (llrd_decay ** num_layers)
    embed_params = [(n, p) for n, p in model.encoder.named_parameters() if 'embeddings' in n]
    optimizer_params.append({
        'params': [p for n, p in embed_params if not any(nd in n for nd in no_decay)],
        'lr': embed_lr,
        'weight_decay': weight_decay
    })
    optimizer_params.append({
        'params': [p for n, p in embed_params if any(nd in n for nd in no_decay)],
        'lr': embed_lr,
        'weight_decay': 0.0
    })
    
    # classification head (highest LR)
    head_params = [
        (n, p) for n, p in model.named_parameters()
        if not n.startswith('encoder.')
    ]
    optimizer_params.append({
        'params': [p for n, p in head_params if not any(nd in n for nd in no_decay)],
        'lr': lr,
        'weight_decay': weight_decay
    })
    optimizer_params.append({
        'params': [p for n, p in head_params if any(nd in n for nd in no_decay)],
        'lr': lr,
        'weight_decay': 0.0
    })
    
    # filter empty param groups
    return [g for g in optimizer_params if len(g['params']) > 0]

optimizer_params = get_optimizer_params(model, LR, WEIGHT_DECAY, LLRD_DECAY)
optimizer = torch.optim.AdamW(optimizer_params, lr=LR)

# lr scheduler (cosine with warmup)
total_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
# warmup_steps = int(0.1 * total_steps)
warmup_steps = 0
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
# mixed precision scaler (enabled=True is default, handles NaN/Inf by skipping updates)
scaler = GradScaler('cuda', enabled=USE_AMP)
print(f"total steps: {total_steps} | warmup: {warmup_steps}")

total steps: 4365 | warmup: 0


In [14]:
@torch.no_grad()
def evaluate(model, eval_loader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(eval_loader, desc="evaluating"):
        batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
        labels = batch.pop('labels')

        with autocast('cuda', enabled=USE_AMP):
            outputs = model(**batch)
        preds = torch.argmax(outputs['logits'], dim=1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    return {"accuracy": acc, "f1_macro": f1_macro}

In [15]:
best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)  # zero at epoch start

    for step, batch in enumerate(pbar):
        batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
        labels = batch.pop('labels')

        # mixed precision forward
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(**batch, labels=labels)
            loss = outputs['loss'] / GRAD_ACCUM_STEPS

        # scaled backward
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            # unscale for gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        pbar.set_postfix({"loss": f"{loss.item() * GRAD_ACCUM_STEPS:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

    if (step + 1) % GRAD_ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    avg_loss = total_loss / len(train_loader)

    print("evaluating")
    metrics = evaluate(model, eval_loader)

    print(f"epoch {epoch+1}/{EPOCHS} | loss: {avg_loss:.4f} | accuracy: {metrics['accuracy']:.4f} | f1: {metrics['f1_macro']:.4f}")

    if metrics["f1_macro"] > best_f1:
        best_f1 = metrics["f1_macro"]
        torch.save(model.state_dict(), "arabert-best.bin")
        print("new best! saved checkpoint.")

    print("-" * 50)

epoch 1/5: 100%|██████████| 1746/1746 [15:08<00:00,  1.92it/s, loss=2.0980, lr=1.03e-05]


evaluating


evaluating: 100%|██████████| 874/874 [04:19<00:00,  3.37it/s]


epoch 1/5 | loss: 2.5673 | accuracy: 0.5318 | f1: 0.3310
new best! saved checkpoint.
--------------------------------------------------


epoch 2/5: 100%|██████████| 1746/1746 [15:10<00:00,  1.92it/s, loss=1.8880, lr=7.45e-06]


...

epoch 4/5 | loss: 1.7381 | accuracy: 0.7057 | f1: 0.5248
new best! saved checkpoint.
--------------------------------------------------


epoch 5/5: 100%|██████████| 1746/1746 [15:09<00:00,  1.92it/s, loss=1.3703, lr=0.00e+00]


evaluating


evaluating: 100%|██████████| 874/874 [04:19<00:00,  3.37it/s]


epoch 5/5 | loss: 1.6672 | accuracy: 0.7115 | f1: 0.5340
new best! saved checkpoint.
--------------------------------------------------


In [16]:
# state_dict = torch.load("arabert-best.bin")
# model.load_state_dict(state_dict)
# model.eval()

In [17]:
# num_labels = len(label2desc)
# model = MedicalClassifier(
#     model_name,
#     num_labels,
#     num_dropouts=NUM_DROPOUTS,
#     dropout_rate=DROPOUT_RATE
# )
# model.

In [18]:
test_ds = CustomDataset(
    model_name,
    test['text'].tolist(),
    max_length=MAX_LENGTH
)
test_loader = DataLoader(
    test_ds, 
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

@torch.no_grad()
def predict(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in tqdm(test_loader):
        batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
        logits = model(**batch)['logits']
        probs = F.softmax(logits, dim=1)
        confs, preds = torch.max(probs, dim=1)
        # pred = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().tolist())

    return all_preds

In [19]:
preds = predict(model, test_loader)

100%|██████████| 583/583 [02:49<00:00,  3.43it/s]


In [20]:
# preds

In [21]:
test['Predicted'] = preds
test.drop('text', inplace=True, axis=1)
test

,Id,Predicted
0,0,74
1,1,27
2,2,40
3,3,22
4,4,36
...,...,...
18629,18629,51
18630,18630,67
18631,18631,70
18632,18632,50


In [22]:
test.to_csv('submission.csv', index=False)

In [23]:
# # arabert 2e-5
# epoch 1/5: 100%|██████████| 1621/1621 [10:56<00:00,  2.47it/s, loss=2.7304, lr=1.10e-05]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:12<00:00,  4.91it/s]
# epoch 1/5 | loss: 2.9407 | accuracy: 0.4765 | f1: 0.2918
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 2/5: 100%|██████████| 1621/1621 [10:50<00:00,  2.49it/s, loss=2.2762, lr=8.53e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:12<00:00,  4.94it/s]
# epoch 2/5 | loss: 2.1450 | accuracy: 0.5200 | f1: 0.3413
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 3/5: 100%|██████████| 1621/1621 [10:55<00:00,  2.47it/s, loss=1.6847, lr=4.69e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:12<00:00,  4.96it/s]
# epoch 3/5 | loss: 1.9151 | accuracy: 0.5230 | f1: 0.3673
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 4/5: 100%|██████████| 1621/1621 [10:55<00:00,  2.47it/s, loss=1.8205, lr=1.32e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:12<00:00,  4.94it/s]
# epoch 4/5 | loss: 1.7512 | accuracy: 0.5390 | f1: 0.3812
# new best! saved checkpoint.
# --------------------------------------------------

In [24]:
# mdeberta v3
# epoch 1/5: 100%|██████████| 1621/1621 [17:27<00:00,  1.55it/s, loss=2.2282, lr=1.10e-05]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:20<00:00,  3.07it/s]
# epoch 1/5 | loss: 3.5654 | accuracy: 0.4110 | f1: 0.2201
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 2/5: 100%|██████████| 1621/1621 [17:25<00:00,  1.55it/s, loss=2.4471, lr=8.53e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:20<00:00,  3.10it/s]
# epoch 2/5 | loss: 2.4759 | accuracy: 0.4555 | f1: 0.2828
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 3/5: 100%|██████████| 1621/1621 [17:23<00:00,  1.55it/s, loss=2.1074, lr=4.69e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:20<00:00,  3.10it/s]
# epoch 3/5 | loss: 2.2426 | accuracy: 0.4905 | f1: 0.3065
# new best! saved checkpoint.
# --------------------------------------------------
# epoch 4/5: 100%|██████████| 1621/1621 [17:26<00:00,  1.55it/s, loss=2.6166, lr=1.32e-06]
# evaluating
# evaluating: 100%|██████████| 63/63 [00:20<00:00,  3.10it/s]
# epoch 4/5 | loss: 2.0950 | accuracy: 0.5030 | f1: 0.3317
# new best! saved checkpoint.
# --------------------------------------------------